# S2 - Protocolos temporais e sensibilidade

Este notebook lê os relatórios strict e pilot já executados, usando o
mesmo índice cacheado. Ele não recalcula o corpus, não treina modelos e
não sela nem aprova nenhum candidato.


In [1]:
from pathlib import Path
import json
import sys

import polars as pl

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'dataset' / 'processed' / 'complaints.parquet').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find project root')
    PROJECT_ROOT = parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.config import ProjectPaths
from consumer_complaint_intelligence.temporal_split import build_criteria_scenarios

paths = ProjectPaths.from_root(PROJECT_ROOT)
index_path = paths.temp_dir / 's2' / 'modeling_index.parquet'
if not index_path.exists():
    raise FileNotFoundError(f'Modeling index not found: {index_path}')
report_paths = {
    'strict': paths.temp_dir / 's2' / 's2_report.json',
    'pilot': paths.temp_dir / 's2' / 's2_report_pilot.json',
}

def load_cached_report(path):
    """Read one cached report envelope without recomputing the corpus."""
    payload = json.loads(path.read_text(encoding='utf-8'))
    return payload.get('report', payload)

reports = {name: load_cached_report(path) for name, path in report_paths.items()}
criteria_scenarios = build_criteria_scenarios()
print({
    'index': index_path.relative_to(PROJECT_ROOT).as_posix(),
    'scenarios': list(reports),
})


{'index': 'temp/s2/modeling_index.parquet', 'scenarios': ['strict', 'pilot']}


In [2]:
candidate_matrix = pl.DataFrame([
    {
        'scenario': scenario,
        'candidate': item['candidate']['name'],
        'status': item['candidate_status'],
        'eligible_classes': item['eligible_class_count'],
        'min_test_novel_groups': item['min_novel_unique_groups_test'],
        'test_purge_pct': round(item['test_purge_pct'], 5),
    }
    for scenario, cached in reports.items()
    for item in cached['candidates']
])
display(candidate_matrix)
status_focus = pl.DataFrame([
    {
        'scenario': 'strict',
        'status': reports['strict']['recommendation_status'],
        'recommended_candidate': reports['strict']['recommended_candidate'],
    },
    {
        'scenario': 'pilot',
        'status': reports['pilot']['recommendation_status'],
        'recommended_candidate': reports['pilot']['recommended_candidate'],
    },
])
display(status_focus)
assert reports['strict']['recommendation_status'] == 'BLOCKED'
assert reports['pilot']['recommendation_status'] == 'READY_FOR_REVIEW'
assert reports['pilot']['recommended_candidate'] == 'post_2023_taxonomy'


scenario,candidate,status,eligible_classes,min_test_novel_groups,test_purge_pct
str,str,str,i64,i64,f64
"""strict""","""historical_stress""","""FAIL""",8,1541,11.16881
"""strict""","""post_2023_taxonomy""","""FAIL""",8,1398,18.57695
"""strict""","""extended_history""","""FAIL""",8,2569,15.49585
"""pilot""","""historical_stress""","""FAIL""",8,1541,11.16881
"""pilot""","""post_2023_taxonomy""","""PASS""",9,1398,18.57695
"""pilot""","""extended_history""","""FAIL""",8,2569,15.49585


scenario,status,recommended_candidate
str,str,str
"""strict""","""BLOCKED""",null
"""pilot""","""READY_FOR_REVIEW""","""post_2023_taxonomy"""


In [3]:
pilot_report = reports['pilot']
selected = next(
    item for item in pilot_report['candidates']
    if item['candidate']['name'] == pilot_report['recommended_candidate']
)
support = pl.DataFrame([
    {
        'partition': partition['partition'],
        **family_support,
    }
    for partition in selected['partitions']
    for family_support in partition['support_by_family']
])
display(support)
print({
    'status': 'NOT SEALED / NOT APPROVED',
    'pilot_candidate': pilot_report['recommended_candidate'],
})


partition,all_text_rows,family,novel_text_rows,novel_unique_groups,purged_seen_before_rows,repeated_within_partition_rows,seen_before_rows,unique_groups
str,i64,str,i64,i64,i64,i64,i64,i64
"""train""",38242,"""cards_prepaid""",38242,31076,0,7166,0,31076
"""train""",10990,"""consumer_lending""",10990,10950,0,40,0,10950
"""train""",415816,"""credit_reporting""",415816,214377,0,201439,0,214377
"""train""",49524,"""debt_collection""",49524,36843,0,12681,0,36843
"""train""",968,"""debt_credit_management""",968,959,0,9,0,959
…,…,…,…,…,…,…,…,…
"""monitor""",932,"""debt_credit_management""",926,920,6,10,6,922
"""monitor""",18678,"""deposit_accounts""",18667,18643,11,29,11,18649
"""monitor""",8411,"""money_services""",8360,8358,51,34,51,8377


{'status': 'NOT SEALED / NOT APPROVED', 'pilot_candidate': 'post_2023_taxonomy'}


## Leitura metodológica

O gate strict mantém 2.000 linhas e 1.000 grupos únicos no treino. A
sensibilidade pilot usa 750 linhas e 750 grupos no treino, mantendo 500
grupos novos em validation e test. Ela é exploratória: não relaxa
retroativamente o gate strict e não aprova o candidato.

No candidato pilot recomendado, `debt_credit_management` tem 968 linhas e
959 grupos no treino, 892 grupos novos em validation e 1.398 grupos novos
em test. Isso supera o limiar pilot de 750, mas exige uma learning curve
na S3 antes de qualquer decisão de suficiência científica.

Qualquer aprovação deverá ocorrer antes do treino, ser explicitamente
revisada e permanecer congelada para a avaliação científica.
